# Tests de mini-funcionalidades de filter_trips

Este notebook se usa para probar helpers y bloques internos de `filter_trips()` antes de hacer tests más integrados de la función completa.

Objetivo:
- verificar minifuncionalidades de forma aislada
- detectar errores de implementación temprano
- dejar una base fácil de portar después a pytest

Convenciones:
- los tests usan `assert`
- cuando una prueba necesita inspección visual, se acompaña con `display(...)`
- las pruebas de este notebook no reemplazan tests integrados

## Bloque 0. Preparación

### Imports generales

In [1]:
from copy import deepcopy
from dataclasses import asdict

import numpy as np
import pandas as pd
import h3

### Imports del modulo

In [2]:
from pylondrina.datasets import TripDataset
from pylondrina.errors import FilterError
from pylondrina.reports import Issue
from pylondrina.schema import DomainSpec, FieldSpec, TripSchema, TripSchemaEffective
from pylondrina.transforms.filtering import (
    FilterOptions,
    TimeFilter,
    build_filter_summary,
    _normalize_filter_request,
    _build_where_mask,
    _build_time_mask,
    _build_spatial_mask,
    _combine_filter_masks,
    _materialize_filtered_tripdataset,
    _truncate_issues_with_limit,
    _build_issues_summary,
    _normalize_where_field_clause,
    _allowed_ops_for_dtype,
    _validate_where_operator_value,
    _normalize_bbox_or_abort,
    _normalize_polygon_or_abort,
    _normalize_h3_cells_or_abort,
    _normalize_iso_timestamp_or_abort,
    _json_safe_scalar,
    _to_json_serializable_or_none,
)

### Helpers de apoyo para test

In [3]:
def h3_from_latlon(lat: float, lon: float, res: int = 8) -> str:
    if hasattr(h3, "latlng_to_cell"):
        return h3.latlng_to_cell(lat, lon, res)
    # fallback por si la librería local usa API antigua
    return h3.geo_to_h3(lat, lon, res)


def assert_issue_codes(issues: list[Issue], expected_codes: list[str]) -> None:
    observed = [issue.code for issue in issues]
    assert observed == expected_codes, f"Esperados={expected_codes}, observados={observed}"


def kept_ids(mask: pd.Series, trips: TripDataset) -> list[str]:
    return trips.data.loc[mask, "movement_id"].tolist()


def make_filter_test_schema() -> TripSchema:
    fields = {
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "mode": FieldSpec(
            name="mode",
            dtype="categorical",
            domain=DomainSpec(values=["bus", "metro", "car", "walk"], extendable=True),
        ),
        "purpose": FieldSpec(
            name="purpose",
            dtype="categorical",
            domain=DomainSpec(values=["work", "study", "leisure"], extendable=True),
        ),
        "distance_km": FieldSpec(name="distance_km", dtype="float"),
        "is_peak": FieldSpec(name="is_peak", dtype="bool"),
        "origin_time_utc": FieldSpec(name="origin_time_utc", dtype="datetime"),
        "destination_time_utc": FieldSpec(name="destination_time_utc", dtype="datetime"),
        "origin_longitude": FieldSpec(name="origin_longitude", dtype="float"),
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float"),
        "destination_longitude": FieldSpec(name="destination_longitude", dtype="float"),
        "destination_latitude": FieldSpec(name="destination_latitude", dtype="float"),
        "origin_h3_index": FieldSpec(name="origin_h3_index", dtype="string"),
        "destination_h3_index": FieldSpec(name="destination_h3_index", dtype="string"),
    }
    return TripSchema(
        version="test-1.0",
        fields=fields,
        required=["movement_id", "user_id"],
    )


def make_filter_test_dataframe() -> pd.DataFrame:
    rows = [
        # m0: completamente dentro del bbox/polygon, tiempo AM
        {
            "movement_id": "m0",
            "user_id": "u0",
            "mode": "bus",
            "purpose": "work",
            "distance_km": 5.0,
            "is_peak": True,
            "origin_time_utc": "2026-01-01T07:10:00Z",
            "destination_time_utc": "2026-01-01T07:40:00Z",
            "origin_longitude": -70.66,
            "origin_latitude": -33.45,
            "destination_longitude": -70.65,
            "destination_latitude": -33.44,
        },
        # m1: también dentro, tiempo AM
        {
            "movement_id": "m1",
            "user_id": "u1",
            "mode": "metro",
            "purpose": "study",
            "distance_km": 12.5,
            "is_peak": True,
            "origin_time_utc": "2026-01-01T08:00:00Z",
            "destination_time_utc": "2026-01-01T08:35:00Z",
            "origin_longitude": -70.64,
            "origin_latitude": -33.46,
            "destination_longitude": -70.63,
            "destination_latitude": -33.45,
        },
        # m2: fuera del bbox/polygon, borde temporal
        {
            "movement_id": "m2",
            "user_id": "u2",
            "mode": "car",
            "purpose": "work",
            "distance_km": 1.2,
            "is_peak": False,
            "origin_time_utc": "2026-01-01T09:30:00Z",
            "destination_time_utc": "2026-01-01T10:00:00Z",
            "origin_longitude": -70.80,
            "origin_latitude": -33.60,
            "destination_longitude": -70.79,
            "destination_latitude": -33.59,
        },
        # m3: origen dentro, destino fuera
        {
            "movement_id": "m3",
            "user_id": "u3",
            "mode": "walk",
            "purpose": None,
            "distance_km": 0.4,
            "is_peak": False,
            "origin_time_utc": "2026-01-01T10:00:00Z",
            "destination_time_utc": "2026-01-01T10:20:00Z",
            "origin_longitude": -70.66,
            "origin_latitude": -33.45,
            "destination_longitude": -70.81,
            "destination_latitude": -33.61,
        },
        # m4: completamente fuera
        {
            "movement_id": "m4",
            "user_id": "u4",
            "mode": "bus",
            "purpose": "leisure",
            "distance_km": 25.0,
            "is_peak": True,
            "origin_time_utc": "2026-01-01T11:00:00Z",
            "destination_time_utc": "2026-01-01T11:30:00Z",
            "origin_longitude": -70.90,
            "origin_latitude": -33.70,
            "destination_longitude": -70.91,
            "destination_latitude": -33.71,
        },
    ]
    df = pd.DataFrame(rows)

    df["origin_time_utc"] = pd.to_datetime(df["origin_time_utc"], utc=True)
    df["destination_time_utc"] = pd.to_datetime(df["destination_time_utc"], utc=True)

    df["origin_h3_index"] = [
        h3_from_latlon(lat, lon, 8)
        for lat, lon in zip(df["origin_latitude"], df["origin_longitude"])
    ]
    df["destination_h3_index"] = [
        h3_from_latlon(lat, lon, 8)
        for lat, lon in zip(df["destination_latitude"], df["destination_longitude"])
    ]
    return df


def make_filter_test_tripdataset(*, validated: bool = True) -> TripDataset:
    df = make_filter_test_dataframe()
    schema = make_filter_test_schema()

    metadata = {
        "dataset_id": "ds_filter_small",
        "is_validated": validated,
        "temporal": {
            "tier": "tier_1",
            "fields_present": ["origin_time_utc", "destination_time_utc"],
        },
        "h3": {
            "resolution": 8,
            "derived_fields": ["origin_h3_index", "destination_h3_index"],
        },
        "schema": {"schema_version": schema.version},
        "domains_effective": {
            "mode": {"values": ["bus", "metro", "car", "walk"]},
            "purpose": {"values": ["work", "study", "leisure"]},
        },
        "events": [
            {
                "op": "import_trips",
                "ts_utc": "2026-04-03T12:00:00Z",
                "parameters": {},
                "summary": {"rows_in": len(df), "rows_out": len(df)},
                "issues_summary": {
                    "counts": {"info": 0, "warning": 0, "error": 0},
                    "top_codes": [],
                },
            }
        ],
    }

    schema_effective = TripSchemaEffective(
        dtype_effective={
            "movement_id": "string",
            "user_id": "string",
            "mode": "categorical",
            "purpose": "categorical",
            "distance_km": "float",
            "is_peak": "bool",
            "origin_time_utc": "datetime",
            "destination_time_utc": "datetime",
            "origin_longitude": "float",
            "origin_latitude": "float",
            "destination_longitude": "float",
            "destination_latitude": "float",
            "origin_h3_index": "string",
            "destination_h3_index": "string",
        },
        domains_effective=deepcopy(metadata["domains_effective"]),
        temporal={"tier": "tier_1"},
        fields_effective=list(df.columns),
    )

    return TripDataset(
        data=df,
        schema=schema,
        schema_version=schema.version,
        provenance={"source": {"name": "synthetic_filter_tests"}},
        field_correspondence={},
        value_correspondence={},
        metadata=metadata,
        schema_effective=schema_effective,
    )


def make_filter_test_tripdataset_tier2() -> TripDataset:
    trips = make_filter_test_tripdataset()
    trips.metadata["temporal"] = {
        "tier": "tier_2",
        "fields_present": ["origin_time_local_hhmm", "destination_time_local_hhmm"],
    }
    return trips


def make_filter_test_tripdataset_missing_destination_coords() -> TripDataset:
    trips = make_filter_test_tripdataset()
    trips.data = trips.data.drop(columns=["destination_longitude", "destination_latitude"])
    return trips

### Configuración visual

In [4]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

print("Imports OK")

Imports OK


## Bloque 1. Utilidades internas de uso general

Este bloque prueba primero las piezas más básicas y reutilizadas: normalización JSON-safe, DSL where, validación de valores por operador, normalización de geometrías/timestamps, truncamiento e issues_summary. Dado el contrato cerrado, estas piezas son críticas porque sostienen parameters, Issue.details, truncamiento y varias precondiciones fatales.

### Test 1.1 - _json_safe_scalar y _to_json_serializable_or_none

Qué prueba: serialización estable de escalares y estructuras anidadas para Issue.details y parameters.

In [5]:
ts = pd.Timestamp("2026-01-01T07:00:00Z")

assert _json_safe_scalar(None) is None
assert _json_safe_scalar(np.nan) is None
assert _json_safe_scalar(ts).startswith("2026-01-01T07:00:00")
assert _json_safe_scalar(True) is True

payload = {
    "a": 1,
    "b": [ts, np.nan, {"x": (1, 2), "y": None}],
}

serialized = _to_json_serializable_or_none(payload)

assert serialized["a"] == 1
assert serialized["b"][0].startswith("2026-01-01T07:00:00")
assert serialized["b"][1] is None
assert serialized["b"][2]["x"] == [1, 2]
assert serialized["b"][2]["y"] is None

### Test 1.2 - _normalize_where_field_clause

Qué prueba: las tres formas válidas del DSL (scalar, sequence, mapping) y el rechazo explícito de set.

In [6]:
clause, shape = _normalize_where_field_clause("work")
assert clause == {"eq": "work"}
assert shape == "scalar"

clause, shape = _normalize_where_field_clause(["bus", "metro"])
assert clause == {"in": ["bus", "metro"]}
assert shape == "sequence"

clause, shape = _normalize_where_field_clause(("bus", "metro"))
assert clause == {"in": ["bus", "metro"]}
assert shape == "sequence"

clause, shape = _normalize_where_field_clause({"gte": 1.0, "lt": 10.0})
assert clause == {"gte": 1.0, "lt": 10.0}
assert shape == "mapping"

clause, shape = _normalize_where_field_clause({"work", "study"})
assert clause is None
assert shape == "set"

### Test 1.3 - _allowed_ops_for_dtype y _validate_where_operator_value

Qué prueba: que la matriz dtype ↔ operador y la forma esperada del valor sigan el cierre concreto.

In [7]:
ops_cat = _allowed_ops_for_dtype("categorical")
assert "eq" in ops_cat
assert "in" in ops_cat
assert "gt" not in ops_cat

ops_num = _allowed_ops_for_dtype("float")
assert "between" in ops_num
assert "gte" in ops_num

ok, msg = _validate_where_operator_value("eq", "work", "categorical")
assert ok is True

ok, msg = _validate_where_operator_value("gt", 3.5, "float")
assert ok is True

ok, msg = _validate_where_operator_value("gt", "3.5", "float")
assert ok is False

ok, msg = _validate_where_operator_value("between", [1.0, 5.0], "float")
assert ok is True

ok, msg = _validate_where_operator_value("between", [1.0], "float")
assert ok is False

ok, msg = _validate_where_operator_value("is_null", True, "string")
assert ok is True

ok, msg = _validate_where_operator_value("is_null", False, "string")
assert ok is False

ok, msg = _validate_where_operator_value("eq", "2026-01-01T07:00:00Z", "datetime")
assert ok is True

ok, msg = _validate_where_operator_value("eq", pd.Timestamp("2026-01-01T07:00:00Z"), "datetime")
assert ok is False

### Test 1.4 - _normalize_iso_timestamp_or_abort

Qué prueba: normalización a UTC con Z y abort fatídico cuando el timestamp no es interpretable.

In [8]:
issues = []
value = "2026-01-01T08:00:00-03:00"
normalized = _normalize_iso_timestamp_or_abort(
    value,
    issues,
    code="FLT.TIME.INVALID_RANGE",
    predicate="overlaps",
    value_role="start",
)
assert normalized == "2026-01-01T11:00:00Z"
assert issues == []

issues = []
try:
    _normalize_iso_timestamp_or_abort(
        12345,
        issues,
        code="FLT.TIME.INVALID_RANGE",
        predicate="overlaps",
        value_role="end",
    )
    raise AssertionError("Debió abortar por timestamp no interpretable")
except ValueError:
    pass

assert len(issues) == 1
assert issues[0].code == "FLT.TIME.INVALID_RANGE"

### Test 1.5 - _normalize_bbox_or_abort

Qué prueba: normalización canónica de bbox y abort cuando la geometría está mal formada.

In [9]:
issues = []
bbox = _normalize_bbox_or_abort([-70.7, -33.5, -70.6, -33.4], issues)
assert bbox == (-70.7, -33.5, -70.6, -33.4)
assert issues == []

issues = []
try:
    _normalize_bbox_or_abort([-70.7, -33.5, -70.8, -33.4], issues)
    raise AssertionError("Debió abortar por bbox invertido")
except ValueError:
    pass

assert len(issues) == 1
assert issues[0].code == "FLT.SPATIAL.INVALID_BBOX"

### Test 1.6 - _normalize_polygon_or_abort

Qué prueba: aceptación de polígono mínimo válido y abort para geometría ilegible.

In [10]:
issues = []
polygon = _normalize_polygon_or_abort(
    [(-70.7, -33.5), (-70.6, -33.5), (-70.6, -33.4), (-70.7, -33.4)],
    issues,
)
assert isinstance(polygon, list)
assert len(polygon) == 4
assert issues == []

issues = []
try:
    _normalize_polygon_or_abort([(-70.7, -33.5), (-70.6, -33.5)], issues)
    raise AssertionError("Debió abortar por polygon con menos de 3 vértices")
except ValueError:
    pass

assert len(issues) == 1
assert issues[0].code == "FLT.SPATIAL.INVALID_POLYGON"

### Test 1.7 - _normalize_h3_cells_or_abort

Qué prueba: deduplicación preservando orden y abort ante valores H3 inválidos.

In [11]:
trips = make_filter_test_tripdataset()
valid_h3_a = trips.data.loc[0, "origin_h3_index"]
valid_h3_b = trips.data.loc[1, "origin_h3_index"]

issues = []
cells = _normalize_h3_cells_or_abort([valid_h3_a, valid_h3_a, valid_h3_b], issues)
assert cells == [valid_h3_a, valid_h3_b]
assert issues == []

issues = []
try:
    _normalize_h3_cells_or_abort([valid_h3_a, "NOT_A_VALID_H3"], issues)
    raise AssertionError("Debió abortar por H3 inválido")
except ValueError:
    pass

assert len(issues) == 1
assert issues[0].code == "FLT.SPATIAL.INVALID_H3_CELLS"

### Test 1.8 - _truncate_issues_with_limit y _build_issues_summary

Qué prueba: truncamiento contractual y resumen compacto por severidad/código.

In [17]:
issues_all = [
    Issue(level="info", code="FLT.INFO.WHERE_APPLIED", message="i1"),
    Issue(level="warning", code="FLT.OUTPUT.EMPTY_RESULT", message="w1"),
    Issue(level="error", code="FLT.TIME.UNSUPPORTED_TIER", message="e1"),
    Issue(level="error", code="FLT.TIME.UNSUPPORTED_TIER", message="e2"),
]

retained, limits = _truncate_issues_with_limit(issues_all, max_issues=3)

assert len(retained) == 3
assert retained[-1].code == "FLT.LIMIT.ISSUES_TRUNCATED"
assert limits["max_issues"] == 3
assert limits["issues_truncated"] is True
assert limits["n_issues_detected_total"] == 4

issues_summary = _build_issues_summary(retained)
assert issues_summary["counts"] == {"info": 1, "warning": 2, "error": 0}
assert any(
    item["code"] == "FLT.LIMIT.ISSUES_TRUNCATED"
    for item in issues_summary["top_codes"]
)

## Bloque 2. Helpers principales del pipeline

Aquí ya se prueban directamente los helpers que definimos para OP-05: request, where, tiempo, espacial, combinación, materialización y summary. La idea es cubrir el caso principal correcto, degradaciones controladas, bordes de contrato y efectos sobre metadata/estado, antes de pasar a integración de la función pública.

### Test 2.1 - _normalize_filter_request caso feliz

Qué prueba: normalización canónica del request, incluyendo time a ISO UTC, bbox, h3_cells y parámetros top-level de evidencia.

In [18]:
trips = make_filter_test_tripdataset()
issues = []

h3_allow = [
    trips.data.loc[0, "origin_h3_index"],
    trips.data.loc[1, "origin_h3_index"],
]

options = FilterOptions(
    where={
        "mode": ["bus", "metro"],
        "distance_km": {"gte": 1.0, "lt": 15.0},
    },
    time=TimeFilter(
        start="2026-01-01T04:00:00-03:00",
        end="2026-01-01T06:00:00-03:00",
        predicate="overlaps",
    ),
    bbox=(-70.7, -33.5, -70.6, -33.4),
    h3_cells=h3_allow,
)

options_eff, parameters, filters_requested = _normalize_filter_request(
    trips,
    options=options,
    max_issues=10,
    sample_rows_per_issue=3,
    issues=issues,
)

assert issues == []
assert filters_requested == ["where", "time", "bbox", "h3_cells"]
assert parameters["time"]["start"] == "2026-01-01T07:00:00Z"
assert parameters["time"]["end"] == "2026-01-01T09:00:00Z"
assert parameters["origin_h3_field"] == "origin_h3_index"
assert parameters["destination_h3_field"] == "destination_h3_index"
assert parameters["max_issues"] == 10
assert parameters["sample_rows_per_issue"] == 3
assert parameters["bbox"] == [-70.7, -33.5, -70.6, -33.4]
assert parameters["h3_cells"] == h3_allow

### Test 2.2 - _normalize_filter_request fatal de configuración

Qué prueba: abort temprano por geometría inválida.

In [19]:
trips = make_filter_test_tripdataset()
issues = []

bad_options = FilterOptions(bbox=(-70.7, -33.5, -70.8, -33.4))

try:
    _normalize_filter_request(
        trips,
        options=bad_options,
        max_issues=10,
        sample_rows_per_issue=3,
        issues=issues,
    )
    raise AssertionError("Debió abortar por bbox inválido")
except ValueError:
    pass

assert len(issues) == 1
assert issues[0].code == "FLT.SPATIAL.INVALID_BBOX"

### Test 2.3 - _build_where_mask caso feliz

Qué prueba: combinación AND entre campos y entre operadores del mismo campo.

In [20]:
trips = make_filter_test_tripdataset()
issues = []

where = {
    "mode": ["bus", "metro"],
    "distance_km": {"gte": 1.0, "lt": 15.0},
    "is_peak": True,
}

mask, applied, omitted = _build_where_mask(
    trips,
    where=where,
    sample_rows_per_issue=3,
    issues=issues,
)

assert applied is True
assert omitted is False
assert kept_ids(mask, trips) == ["m0", "m1"]

codes = [issue.code for issue in issues]
assert "FLT.INFO.WHERE_APPLIED" in codes

### Test 2.4 - _build_where_mask degradado por cláusulas no aplicables

Qué prueba: degradación controlada cuando hay campo inexistente, operador incompatible y valor inválido.

In [21]:
trips = make_filter_test_tripdataset()
issues = []

where = {
    "does_not_exist": "x",
    "mode": {"gt": "bus"},        # incompatible con categorical
    "purpose": {"in": set(["work"])},  # set no aceptado por el DSL
}

mask, applied, omitted = _build_where_mask(
    trips,
    where=where,
    sample_rows_per_issue=3,
    issues=issues,
)

assert mask is None
assert applied is False
assert omitted is True

observed_codes = sorted(issue.code for issue in issues)
expected_codes = sorted([
    "FLT.WHERE.FIELD_NOT_FOUND",
    "FLT.WHERE.OP_INCOMPATIBLE",
    "FLT.WHERE.INVALID_VALUE_SHAPE",
])
assert observed_codes == expected_codes

### Test 2.5 - _build_time_mask Tier 1 feliz

Qué prueba: filtro temporal evaluable sobre tier_1 con predicado overlaps.

In [22]:
trips = make_filter_test_tripdataset()
issues = []

time_filter = TimeFilter(
    start="2026-01-01T07:00:00Z",
    end="2026-01-01T09:00:00Z",
    predicate="overlaps",
)

mask, applied, omitted = _build_time_mask(
    trips,
    time=time_filter,
    sample_rows_per_issue=3,
    issues=issues,
)

assert applied is True
assert omitted is False
assert kept_ids(mask, trips) == ["m0", "m1"]

codes = [issue.code for issue in issues]
assert "FLT.INFO.TIME_APPLIED" in codes

### Test 2.6 - _build_time_mask Tier 2 no evaluable

Qué prueba: issue error retornable y omisión del eje temporal cuando el dataset está en tier_2.

In [23]:
trips = make_filter_test_tripdataset_tier2()
issues = []

time_filter = TimeFilter(
    start="2026-01-01T07:00:00Z",
    end="2026-01-01T09:00:00Z",
    predicate="overlaps",
)

mask, applied, omitted = _build_time_mask(
    trips,
    time=time_filter,
    sample_rows_per_issue=3,
    issues=issues,
)

assert mask is None
assert applied is False
assert omitted is True
assert_issue_codes(issues, ["FLT.TIME.UNSUPPORTED_TIER"])

### Test 2.7 - _build_spatial_mask caso feliz con múltiples subfiltros

Qué prueba: bbox, polygon y h3_cells simultáneos, combinados sobre el mismo eje espacial.

In [24]:
trips = make_filter_test_tripdataset()
issues = []

bbox = (-70.70, -33.50, -70.60, -33.40)
polygon = [(-70.70, -33.50), (-70.60, -33.50), (-70.60, -33.40), (-70.70, -33.40)]
h3_cells = [
    trips.data.loc[0, "origin_h3_index"],
    trips.data.loc[1, "origin_h3_index"],
    trips.data.loc[3, "origin_h3_index"],
]

masks, applied, omitted = _build_spatial_mask(
    trips,
    bbox=bbox,
    polygon=polygon,
    h3_cells=h3_cells,
    spatial_predicate="origin",
    origin_h3_field="origin_h3_index",
    destination_h3_field="destination_h3_index",
    sample_rows_per_issue=3,
    issues=issues,
)

assert applied == ["bbox", "polygon", "h3_cells"]
assert omitted == []

assert kept_ids(masks["bbox"], trips) == ["m0", "m1", "m3"]
assert kept_ids(masks["polygon"], trips) == ["m0", "m1", "m3"]
assert kept_ids(masks["h3_cells"], trips) == ["m0", "m1", "m3"]

codes = [issue.code for issue in issues]
assert "FLT.INFO.BBOX_APPLIED" in codes
assert "FLT.INFO.POLYGON_APPLIED" in codes
assert "FLT.INFO.H3_APPLIED" in codes

### Test 2.8 - _build_spatial_mask degradado por columnas faltantes

Qué prueba: omisión recuperable cuando faltan columnas requeridas para evaluar filtros espaciales.

In [25]:
trips = make_filter_test_tripdataset_missing_destination_coords()
issues = []

bbox = (-70.70, -33.50, -70.60, -33.40)
h3_cells = [trips.data.loc[0, "origin_h3_index"]]

masks, applied, omitted = _build_spatial_mask(
    trips,
    bbox=bbox,
    polygon=None,
    h3_cells=h3_cells,
    spatial_predicate="both",
    origin_h3_field="origin_h3_missing",   # campo declarado, pero ausente
    destination_h3_field="destination_h3_missing",
    sample_rows_per_issue=3,
    issues=issues,
)

assert applied == []
assert omitted == ["bbox", "h3_cells"]

observed_codes = sorted(issue.code for issue in issues)
expected_codes = sorted([
    "FLT.SPATIAL.MISSING_REQUIRED_COLUMNS",
    "FLT.SPATIAL.H3_FIELD_MISSING",
])
assert observed_codes == expected_codes

### Test 2.9 - _combine_filter_masks sin filtros definidos

Qué prueba: caso de “sin cambios efectivos” por ausencia total de filtros.

In [26]:
trips = make_filter_test_tripdataset()
issues = []

mask_survival, dropped_by_filter, rows_in, rows_out, dropped_total = _combine_filter_masks(
    trips,
    mask_items=[],
    filters_requested=[],
    filters_applied=[],
    filters_omitted=[],
    issues=issues,
)

assert rows_in == len(trips.data)
assert rows_out == len(trips.data)
assert dropped_total == 0
assert mask_survival.all()
assert dropped_by_filter["where"] == 0
assert dropped_by_filter["time"] == 0
assert_issue_codes(issues, ["FLT.INFO.NO_FILTERS_DEFINED"])

### Test 2.10 - _combine_filter_masks combinación incremental y resultado vacío

Qué prueba: contabilidad incremental real en dropped_by_filter y caso borde de dataset vacío.

In [ ]:
trips = make_filter_test_tripdataset()
issues = []

where_mask = trips.data["mode"].isin(["bus", "metro", "car"])   # m0,m1,m2
time_mask = trips.data["purpose"].eq("work")                    # m0,m2
h3_mask = pd.Series(False, index=trips.data.index, dtype=bool)  # vacía todo

mask_survival, dropped_by_filter, rows_in, rows_out, dropped_total = _combine_filter_masks(
    trips,
    mask_items=[
        ("where", where_mask),
        ("time", time_mask),
        ("h3_cells", h3_mask),
    ],
    filters_requested=["where", "time", "h3_cells"],
    filters_applied=["where", "time", "h3_cells"],
    filters_omitted=[],
    issues=issues,
)

assert rows_in == 5
assert rows_out == 0
assert dropped_total == 5

# where: cae 1 (m3)
# time: sobre sobrevivientes [m0,m1,m2,m4], caen 2 (m1,m4)
# h3_cells: sobre sobrevivientes [m0,m2], caen 2
print(dropped_by_filter["time"])
display(trips.data)
assert dropped_by_filter["where"] == 1
assert dropped_by_filter["time"] == 2
assert dropped_by_filter["h3_cells"] == 2

assert_issue_codes(issues, ["FLT.OUTPUT.EMPTY_RESULT"])

2


,movement_id,user_id,mode,purpose,distance_km,is_peak,origin_time_utc,destination_time_utc,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index
0,m0,u0,bus,work,5.0,True,2026-01-01 07:10:00+00:00,2026-01-01 07:40:00+00:00,-70.66,-33.45,-70.65,-33.44,88b2c55417fffff,88b2c5541bfffff
1,m1,u1,metro,study,12.5,True,2026-01-01 08:00:00+00:00,2026-01-01 08:35:00+00:00,-70.64,-33.46,-70.63,-33.45,88b2c5548dfffff,88b2c554c1fffff
2,m2,u2,car,work,1.2,False,2026-01-01 09:30:00+00:00,2026-01-01 10:00:00+00:00,-70.80,-33.60,-70.79,-33.59,88b2c54e11fffff,88b2c54525fffff
3,m3,u3,walk,None,0.4,False,2026-01-01 10:00:00+00:00,2026-01-01 10:20:00+00:00,-70.66,-33.45,-70.81,-33.61,88b2c55417fffff,88b2c54e3bfffff
4,m4,u4,bus,leisure,25.0,True,2026-01-01 11:00:00+00:00,2026-01-01 11:30:00+00:00,-70.90,-33.70,-70.91,-33.71,88b2c54da1fffff,88b2c0b29bfffff


### Test 2.11 - _materialize_filtered_tripdataset con keep_metadata=True

Qué prueba: commit único final, nuevo dataset, no mutación del input y preservación de metadata trazable.

In [34]:
trips = make_filter_test_tripdataset(validated=True)
data_before = trips.data.copy(deep=True)

mask_survival = trips.data["mode"].isin(["bus", "metro"])
filtered = _materialize_filtered_tripdataset(
    trips,
    mask_survival=mask_survival,
    keep_metadata=True,
)

assert filtered is not trips
assert filtered.data["movement_id"].tolist() == ["m0", "m1", "m4"]

# no muta el input
pd.testing.assert_frame_equal(trips.data, data_before)

# preservación de estado relevante
assert filtered.metadata["dataset_id"] == trips.metadata["dataset_id"]
assert filtered.metadata["is_validated"] is True
assert "events" in filtered.metadata
assert filtered.metadata["events"] is not trips.metadata["events"]
assert filtered.schema is trips.schema
assert filtered.schema_effective is not trips.schema_effective

### Test 2.12 - _materialize_filtered_tripdataset con keep_metadata=False

Qué prueba: metadata mínima cerrada por contrato, sin copiar historial de eventos.

In [35]:
trips = make_filter_test_tripdataset(validated=True)
trips.metadata["artifact_id"] = "artifact_should_not_survive_here"

mask_survival = trips.data["mode"].isin(["bus", "metro"])
filtered = _materialize_filtered_tripdataset(
    trips,
    mask_survival=mask_survival,
    keep_metadata=False,
)

assert filtered.data["movement_id"].tolist() == ["m0", "m1", "m4"]

expected_keys = {"dataset_id", "is_validated", "temporal", "h3", "schema", "domains_effective"}
assert set(filtered.metadata.keys()) == expected_keys
assert "events" not in filtered.metadata
assert "artifact_id" not in filtered.metadata
assert filtered.metadata["is_validated"] is True

### Test 2.13 - build_filter_summary

Qué prueba: shape estable del summary, claves de filtros completas y bloque limits opcional.

In [36]:
summary = build_filter_summary(
    rows_in=5,
    rows_out=2,
    dropped_by_filter={"where": 2, "time": 1},
    filters_requested=["where", "time", "bbox"],
    filters_applied=["where", "time"],
    filters_omitted=["bbox"],
    limits={
        "max_issues": 3,
        "issues_truncated": True,
        "n_issues_emitted": 3,
        "n_issues_detected_total": 7,
    },
)

assert summary["rows_in"] == 5
assert summary["rows_out"] == 2
assert summary["dropped_total"] == 3
assert summary["filters_requested"] == ["where", "time", "bbox"]
assert summary["filters_applied"] == ["where", "time"]
assert summary["filters_omitted"] == ["bbox"]

# las claves del bloque dropped_by_filter deben existir siempre
assert summary["dropped_by_filter"]["where"] == 2
assert summary["dropped_by_filter"]["time"] == 1
assert summary["dropped_by_filter"]["bbox"] == 0
assert summary["dropped_by_filter"]["polygon"] == 0
assert summary["dropped_by_filter"]["h3_cells"] == 0

assert summary["limits"]["issues_truncated"] is True

## Bloque 3: Smoke tests integrados de filter_trips

### 3.1 - Fixtures mínimas reutilizables

Qué prepara: un par de utilidades pequeñas para inspeccionar el evento final y mantener los tests de smoke legibles.

In [37]:
from copy import deepcopy

import pandas as pd

from pylondrina.errors import FilterError
from pylondrina.transforms.filtering import FilterOptions, TimeFilter, filter_trips


def get_last_event(trips):
    events = trips.metadata.get("events", [])
    assert isinstance(events, list)
    assert len(events) >= 1
    return events[-1]

### Test 3.2 - Happy path mínimo integrado

Qué prueba: el camino feliz completo de la función pública: nuevo dataset, OperationReport, summary, parameters, evento filter_trips, preservación de is_validated y no mutación del input.

In [41]:
trips = make_filter_test_tripdataset(validated=True)
data_before = trips.data.copy(deep=True)
events_before = deepcopy(trips.metadata["events"])
validated_before = trips.metadata["is_validated"]

options = FilterOptions(
    where={"mode": ["bus", "metro"]},
    time=TimeFilter(
        start="2026-01-01T07:00:00Z",
        end="2026-01-01T09:00:00Z",
        predicate="overlaps",
    ),
    bbox=(-70.70, -33.50, -70.60, -33.40),
)

filtered, report = filter_trips(
    trips,
    options=options,
    max_issues=20,
    sample_rows_per_issue=3,
)

# Resultado tabular
assert filtered is not trips
assert filtered.data["movement_id"].tolist() == ["m0", "m1"]

# Reporte
assert report.ok is True
assert report.summary["rows_in"] == 5
assert report.summary["rows_out"] == 2
assert report.summary["dropped_total"] == 3
assert report.summary["filters_requested"] == ["where", "time", "bbox"]
assert report.summary["filters_applied"] == ["where", "time", "bbox"]
assert report.summary["filters_omitted"] == []

# Parameters efectivos
assert report.parameters["max_issues"] == 20
assert report.parameters["sample_rows_per_issue"] == 3
assert report.parameters["origin_h3_field"] == "origin_h3_index"
assert report.parameters["destination_h3_field"] == "destination_h3_index"

# Evento en el resultado
assert len(filtered.metadata["events"]) == len(events_before) + 1
event = get_last_event(filtered)
assert event["op"] == "filter_trips"
assert event["summary"] == report.summary
assert event["parameters"] == report.parameters
assert "issues_summary" in event

# Preservación de estado drop-only
assert filtered.metadata["is_validated"] is validated_before

# El input no se muta
pd.testing.assert_frame_equal(trips.data, data_before)
assert trips.metadata["events"] == events_before
assert trips.metadata["is_validated"] == validated_before

display(trips.data)
display(filtered.data)
display(filtered.metadata["events"])
display(report.issues)
display(report.summary)

,movement_id,user_id,mode,purpose,distance_km,is_peak,origin_time_utc,destination_time_utc,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index
0,m0,u0,bus,work,5.0,True,2026-01-01 07:10:00+00:00,2026-01-01 07:40:00+00:00,-70.66,-33.45,-70.65,-33.44,88b2c55417fffff,88b2c5541bfffff
1,m1,u1,metro,study,12.5,True,2026-01-01 08:00:00+00:00,2026-01-01 08:35:00+00:00,-70.64,-33.46,-70.63,-33.45,88b2c5548dfffff,88b2c554c1fffff
2,m2,u2,car,work,1.2,False,2026-01-01 09:30:00+00:00,2026-01-01 10:00:00+00:00,-70.80,-33.60,-70.79,-33.59,88b2c54e11fffff,88b2c54525fffff
3,m3,u3,walk,None,0.4,False,2026-01-01 10:00:00+00:00,2026-01-01 10:20:00+00:00,-70.66,-33.45,-70.81,-33.61,88b2c55417fffff,88b2c54e3bfffff
4,m4,u4,bus,leisure,25.0,True,2026-01-01 11:00:00+00:00,2026-01-01 11:30:00+00:00,-70.90,-33.70,-70.91,-33.71,88b2c54da1fffff,88b2c0b29bfffff


,movement_id,user_id,mode,purpose,distance_km,is_peak,origin_time_utc,destination_time_utc,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index
0,m0,u0,bus,work,5.0,True,2026-01-01 07:10:00+00:00,2026-01-01 07:40:00+00:00,-70.66,-33.45,-70.65,-33.44,88b2c55417fffff,88b2c5541bfffff
1,m1,u1,metro,study,12.5,True,2026-01-01 08:00:00+00:00,2026-01-01 08:35:00+00:00,-70.64,-33.46,-70.63,-33.45,88b2c5548dfffff,88b2c554c1fffff


[{'op': 'import_trips',
  'ts_utc': '2026-04-03T12:00:00Z',
  'parameters': {},
  'summary': {'rows_in': 5, 'rows_out': 5},
  'issues_summary': {'counts': {'info': 0, 'warning': 0, 'error': 0},
   'top_codes': []}},
 {'op': 'filter_trips',
  'ts_utc': '2026-04-03T22:36:56.470059Z',
  'parameters': {'where': {'mode': ['bus', 'metro']},
   'time': {'start': '2026-01-01T07:00:00Z',
    'end': '2026-01-01T09:00:00Z',
    'predicate': 'overlaps'},
   'bbox': [-70.7, -33.5, -70.6, -33.4],
   'polygon': None,
   'h3_cells': None,
   'spatial_predicate': 'origin',
   'origin_h3_field': 'origin_h3_index',
   'destination_h3_field': 'destination_h3_index',
   'keep_metadata': True,
   'strict': False,
   'max_issues': 20,
   'sample_rows_per_issue': 3},
  'summary': {'rows_in': 5,
   'rows_out': 2,
   'dropped_total': 3,
   'dropped_by_filter': {'where': 2,
    'time': 1,
    'bbox': 0,
    'polygon': 0,
    'h3_cells': 0},
   'filters_requested': ['where', 'time', 'bbox'],
   'filters_applied':

[Issue(level='info', code='FLT.INFO.WHERE_APPLIED', message='Se aplicó la cláusula where y el subconjunto pasó de 5 a 3 filas en este eje.', field=None, source_field=None, row_count=None, details={'check': 'where', 'where_fields': ['mode'], 'rows_in_scope': 5, 'rows_out_scope': 3, 'rows_removed_scope': 2, 'row_indices_sample_removed': [2, 3], 'movement_ids_sample_removed': ['m2', 'm3'], 'values_sample': [{'mode': 'car'}, {'mode': 'walk'}], 'policy': 'and_between_fields_and_ops'}),
 Issue(level='info', code='FLT.INFO.TIME_APPLIED', message="Se aplicó el filtro temporal ('overlaps') y el subconjunto pasó de 5 a 2 filas en este eje.", field=None, source_field=None, row_count=None, details={'check': 'time', 'rows_in_scope': 5, 'rows_out_scope': 2, 'rows_removed_scope': 3, 'row_indices_sample_removed': [2, 3, 4], 'movement_ids_sample_removed': ['m2', 'm3', 'm4'], 'values_sample': [{'origin_time_utc': '2026-01-01T09:30:00+00:00', 'destination_time_utc': '2026-01-01T10:00:00+00:00'}, {'origin

{'rows_in': 5,
 'rows_out': 2,
 'dropped_total': 3,
 'dropped_by_filter': {'where': 2,
  'time': 1,
  'bbox': 0,
  'polygon': 0,
  'h3_cells': 0},
 'filters_requested': ['where', 'time', 'bbox'],
 'filters_applied': ['where', 'time', 'bbox'],
 'filters_omitted': []}

### Test 3.3 - keep_metadata=False y metadata mínima

Qué prueba: la política cerrada de keep_metadata=False: no copiar historial de eventos, no agregar evento nuevo, pero sí preservar metadata operativa mínima y is_validated.

In [ ]:
trips = make_filter_test_tripdataset(validated=True)
data_before = trips.data.copy(deep=True)
events_before = deepcopy(trips.metadata["events"])

options = FilterOptions(
    where={"mode": "bus"},
    keep_metadata=False,
)

filtered, report = filter_trips(
    trips,
    options=options,
)

assert filtered.data["movement_id"].tolist() == ["m0", "m4"]
assert report.ok is True
assert report.summary["rows_in"] == 5
assert report.summary["rows_out"] == 2

# No debe haber historial de events ni evento nuevo
assert "events" not in filtered.metadata

# Metadata mínima preservada
expected_keys = {"dataset_id", "is_validated", "temporal", "h3", "schema", "domains_effective"}
assert set(filtered.metadata.keys()) == expected_keys
assert filtered.metadata["dataset_id"] == trips.metadata["dataset_id"]
assert filtered.metadata["is_validated"] is True

# Input intacto
pd.testing.assert_frame_equal(trips.data, data_before)
assert trips.metadata["events"] == events_before


### Test 3.4 - Degradado no fatal con strict=False

Qué prueba: un eje no evaluable no aborta cuando strict=False; queda issue error, el eje se omite, los otros filtros sí se aplican, se retorna dataset y se registra evento.

In [44]:
trips = make_filter_test_tripdataset_tier2()
data_before = trips.data.copy(deep=True)

options = FilterOptions(
    time=TimeFilter(
        start="2026-01-01T07:00:00Z",
        end="2026-01-01T09:00:00Z",
        predicate="overlaps",
    ),
    bbox=(-70.70, -33.50, -70.60, -33.40),
    strict=False,
)

filtered, report = filter_trips(
    trips,
    options=options,
)

codes = [issue.code for issue in report.issues]

# El tiempo se omite por Tier 2, pero bbox sí se aplica
assert "FLT.TIME.UNSUPPORTED_TIER" in codes
assert "FLT.INFO.BBOX_APPLIED" in codes

assert report.ok is False
assert report.summary["filters_requested"] == ["time", "bbox"]
assert report.summary["filters_applied"] == ["bbox"]
assert report.summary["filters_omitted"] == ["time"]

# bbox sobre origen conserva m0, m1, m3
assert filtered.data["movement_id"].tolist() == ["m0", "m1", "m3"]

# Se preserva is_validated porque sigue siendo drop-only
assert filtered.metadata["is_validated"] == trips.metadata["is_validated"]

# Se registra evento en el resultado
event = get_last_event(filtered)
assert event["op"] == "filter_trips"
assert event["summary"] == report.summary
assert "issues_summary" in event

# Input intacto
pd.testing.assert_frame_equal(trips.data, data_before)

display(trips.data)
display(filtered.data)
display(report.issues)

,movement_id,user_id,mode,purpose,distance_km,is_peak,origin_time_utc,destination_time_utc,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index
0,m0,u0,bus,work,5.0,True,2026-01-01 07:10:00+00:00,2026-01-01 07:40:00+00:00,-70.66,-33.45,-70.65,-33.44,88b2c55417fffff,88b2c5541bfffff
1,m1,u1,metro,study,12.5,True,2026-01-01 08:00:00+00:00,2026-01-01 08:35:00+00:00,-70.64,-33.46,-70.63,-33.45,88b2c5548dfffff,88b2c554c1fffff
2,m2,u2,car,work,1.2,False,2026-01-01 09:30:00+00:00,2026-01-01 10:00:00+00:00,-70.80,-33.60,-70.79,-33.59,88b2c54e11fffff,88b2c54525fffff
3,m3,u3,walk,None,0.4,False,2026-01-01 10:00:00+00:00,2026-01-01 10:20:00+00:00,-70.66,-33.45,-70.81,-33.61,88b2c55417fffff,88b2c54e3bfffff
4,m4,u4,bus,leisure,25.0,True,2026-01-01 11:00:00+00:00,2026-01-01 11:30:00+00:00,-70.90,-33.70,-70.91,-33.71,88b2c54da1fffff,88b2c0b29bfffff


,movement_id,user_id,mode,purpose,distance_km,is_peak,origin_time_utc,destination_time_utc,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index
0,m0,u0,bus,work,5.0,True,2026-01-01 07:10:00+00:00,2026-01-01 07:40:00+00:00,-70.66,-33.45,-70.65,-33.44,88b2c55417fffff,88b2c5541bfffff
1,m1,u1,metro,study,12.5,True,2026-01-01 08:00:00+00:00,2026-01-01 08:35:00+00:00,-70.64,-33.46,-70.63,-33.45,88b2c5548dfffff,88b2c554c1fffff
3,m3,u3,walk,None,0.4,False,2026-01-01 10:00:00+00:00,2026-01-01 10:20:00+00:00,-70.66,-33.45,-70.81,-33.61,88b2c55417fffff,88b2c54e3bfffff


[Issue(level='error', code='FLT.TIME.UNSUPPORTED_TIER', message="No se puede aplicar el filtro temporal: el dataset está en 'tier_2' y OP-05 solo evalúa Tier 1 absoluto; la regla se omitirá.", field=None, source_field=None, row_count=None, details={'check': 'time', 'predicate': 'overlaps', 'start': '2026-01-01T07:00:00Z', 'end': '2026-01-01T09:00:00Z', 'temporal_tier': 'tier_2', 'fields_present': ['origin_time_local_hhmm', 'destination_time_local_hhmm'], 'missing_fields': ['origin_time_utc', 'destination_time_utc'], 'expected': 'tier_1 with origin_time_utc and destination_time_utc', 'observed': 'tier_2', 'reason': 'unsupported_temporal_tier', 'action': 'omit_time_filter'}),
 Issue(level='info', code='FLT.INFO.BBOX_APPLIED', message='Se aplicó el filtro bbox y el subconjunto pasó de 5 a 3 filas en este eje.', field=None, source_field=None, row_count=None, details={'check': 'spatial', 'rows_in_scope': 5, 'rows_out_scope': 3, 'rows_removed_scope': 2, 'row_indices_sample_removed': [2, 4], 

### Test 3.5 - strict=True con error recuperable

Qué prueba: si existe un issue error recuperable y strict=True, la operación escala a FilterError. Como OP-05 no muta el input y el evento vive en el resultado, aquí lo importante es verificar el raise correcto y que el input permanezca intacto.

In [46]:
trips = make_filter_test_tripdataset_tier2()
data_before = trips.data.copy(deep=True)
events_before = deepcopy(trips.metadata["events"])
validated_before = trips.metadata["is_validated"]

raised = None
try:
    filter_trips(
        trips,
        options=FilterOptions(
            time=TimeFilter(
                start="2026-01-01T07:00:00Z",
                end="2026-01-01T09:00:00Z",
                predicate="overlaps",
            ),
            strict=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, FilterError)
assert raised.issue is not None
assert raised.issue.code == "FLT.TIME.UNSUPPORTED_TIER"

# El input no cambia
pd.testing.assert_frame_equal(trips.data, data_before)
assert trips.metadata["events"] == events_before
assert trips.metadata["is_validated"] == validated_before

display(raised.issues)

(Issue(level='error', code='FLT.TIME.UNSUPPORTED_TIER', message="No se puede aplicar el filtro temporal: el dataset está en 'tier_2' y OP-05 solo evalúa Tier 1 absoluto; la regla se omitirá.", field=None, source_field=None, row_count=None, details={'check': 'time', 'predicate': 'overlaps', 'start': '2026-01-01T07:00:00Z', 'end': '2026-01-01T09:00:00Z', 'temporal_tier': 'tier_2', 'fields_present': ['origin_time_local_hhmm', 'destination_time_local_hhmm'], 'missing_fields': ['origin_time_utc', 'destination_time_utc'], 'expected': 'tier_1 with origin_time_utc and destination_time_utc', 'observed': 'tier_2', 'reason': 'unsupported_temporal_tier', 'action': 'omit_time_filter'}),
 Issue(level='info', code='FLT.INFO.FILTER_WITHOUT_EFFECT', message='Los filtros evaluados no produjeron cambios efectivos: no se descartaron filas.', field=None, source_field=None, row_count=None, details={'rows_in': 5, 'rows_out': 5, 'dropped_total': 0, 'filters_requested': ['time'], 'filters_applied': [], 'filter

### Test 3.6 - Fatal de configuración sin evento

Qué prueba: abort fatal antes de correr el pipeline normal. En este caso, bbox inválido. No debe haber retorno, ni mutación del input, ni side effects sobre metadata.

In [48]:
trips = make_filter_test_tripdataset()
data_before = trips.data.copy(deep=True)
events_before = deepcopy(trips.metadata["events"])
validated_before = trips.metadata["is_validated"]

raised = None
try:
    filter_trips(
        trips,
        options=FilterOptions(
            bbox=(-70.70, -33.50, -70.80, -33.40),  # min_lon > max_lon
        ),
    )
except Exception as e:
    raised = e

assert raised is not None

# Input completamente intacto
pd.testing.assert_frame_equal(trips.data, data_before)
assert trips.metadata["events"] == events_before
assert trips.metadata["is_validated"] == validated_before

display(raised.issues)

(Issue(level='error', code='FLT.SPATIAL.INVALID_BBOX', message='La geometría bbox no es interpretable como (min_lon, min_lat, max_lon, max_lat).', field=None, source_field=None, row_count=None, details={'check': 'spatial', 'bbox': [-70.7, -33.5, -70.8, -33.4], 'observed': {'min_lon': -70.7, 'min_lat': -33.5, 'max_lon': -70.8, 'max_lat': -33.4}, 'expected': 'tuple/list[float, float, float, float] with min<=max', 'reason': 'invalid_bbox', 'action': 'abort'}),)

### Grupo 3.7 - Resultado vacío retornable

Qué prueba: dataset vacío como caso retornable, con issue warning, ok=True, evento correcto y preservación de is_validated.

In [50]:
trips = make_filter_test_tripdataset(validated=True)

options = FilterOptions(
    where={"mode": "bus"},
    time=TimeFilter(
        start="2026-01-01T09:00:00Z",
        end="2026-01-01T09:15:00Z",
        predicate="overlaps",
    ),
)

filtered, report = filter_trips(
    trips,
    options=options,
)

codes = [issue.code for issue in report.issues]

assert report.ok is True
assert "FLT.OUTPUT.EMPTY_RESULT" in codes

assert report.summary["rows_in"] == 5
assert report.summary["rows_out"] == 0
assert report.summary["dropped_total"] == 5
assert filtered.data.empty is True

event = get_last_event(filtered)
assert event["op"] == "filter_trips"
assert event["summary"] == report.summary

assert filtered.metadata["is_validated"] is True

display(trips.data)
display(filtered.data)
display(report.issues)

,movement_id,user_id,mode,purpose,distance_km,is_peak,origin_time_utc,destination_time_utc,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index
0,m0,u0,bus,work,5.0,True,2026-01-01 07:10:00+00:00,2026-01-01 07:40:00+00:00,-70.66,-33.45,-70.65,-33.44,88b2c55417fffff,88b2c5541bfffff
1,m1,u1,metro,study,12.5,True,2026-01-01 08:00:00+00:00,2026-01-01 08:35:00+00:00,-70.64,-33.46,-70.63,-33.45,88b2c5548dfffff,88b2c554c1fffff
2,m2,u2,car,work,1.2,False,2026-01-01 09:30:00+00:00,2026-01-01 10:00:00+00:00,-70.80,-33.60,-70.79,-33.59,88b2c54e11fffff,88b2c54525fffff
3,m3,u3,walk,None,0.4,False,2026-01-01 10:00:00+00:00,2026-01-01 10:20:00+00:00,-70.66,-33.45,-70.81,-33.61,88b2c55417fffff,88b2c54e3bfffff
4,m4,u4,bus,leisure,25.0,True,2026-01-01 11:00:00+00:00,2026-01-01 11:30:00+00:00,-70.90,-33.70,-70.91,-33.71,88b2c54da1fffff,88b2c0b29bfffff


,movement_id,user_id,mode,purpose,distance_km,is_peak,origin_time_utc,destination_time_utc,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index


[Issue(level='info', code='FLT.INFO.WHERE_APPLIED', message='Se aplicó la cláusula where y el subconjunto pasó de 5 a 2 filas en este eje.', field=None, source_field=None, row_count=None, details={'check': 'where', 'where_fields': ['mode'], 'rows_in_scope': 5, 'rows_out_scope': 2, 'rows_removed_scope': 3, 'row_indices_sample_removed': [1, 2, 3], 'movement_ids_sample_removed': ['m1', 'm2', 'm3'], 'values_sample': [{'mode': 'metro'}, {'mode': 'car'}, {'mode': 'walk'}], 'policy': 'and_between_fields_and_ops'}),
 Issue(level='info', code='FLT.INFO.TIME_APPLIED', message="Se aplicó el filtro temporal ('overlaps') y el subconjunto pasó de 5 a 0 filas en este eje.", field=None, source_field=None, row_count=None, details={'check': 'time', 'rows_in_scope': 5, 'rows_out_scope': 0, 'rows_removed_scope': 5, 'row_indices_sample_removed': [0, 1, 2, 3, 4], 'movement_ids_sample_removed': ['m0', 'm1', 'm2', 'm3', 'm4'], 'values_sample': [{'origin_time_utc': '2026-01-01T07:10:00+00:00', 'destination_tim

### Grupo 3.8 - Truncamiento + consistencia summary / evento

Qué prueba: truncamiento explícito del reporte, issue final FLT.LIMIT.ISSUES_TRUNCATED, bloque limits en summary, y consistencia entre report y evento.

In [52]:
trips = make_filter_test_tripdataset_tier2()

options = FilterOptions(
    where={
        "does_not_exist": "x",            # FIELD_NOT_FOUND
        "mode": {"gt": "bus"},            # OP_INCOMPATIBLE
        "purpose": {"in": set(["work"])}, # INVALID_VALUE_SHAPE
    },
    time=TimeFilter(
        start="2026-01-01T07:00:00Z",
        end="2026-01-01T09:00:00Z",
        predicate="overlaps",             # TIME.UNSUPPORTED_TIER
    ),
    strict=False,
)

filtered, report = filter_trips(
    trips,
    options=options,
    max_issues=2,
)

codes = [issue.code for issue in report.issues]

assert "FLT.LIMIT.ISSUES_TRUNCATED" in codes
assert report.summary["limits"]["issues_truncated"] is True
assert report.summary["limits"]["max_issues"] == 2
assert report.summary["limits"]["n_issues_emitted"] <= 2
assert report.summary["limits"]["n_issues_detected_total"] >= report.summary["limits"]["n_issues_emitted"]

event = get_last_event(filtered)
assert event["summary"] == report.summary
assert event["parameters"] == report.parameters
assert "issues_summary" in event

display(trips.data)
display(filtered.data)
display(report.issues)

,movement_id,user_id,mode,purpose,distance_km,is_peak,origin_time_utc,destination_time_utc,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index
0,m0,u0,bus,work,5.0,True,2026-01-01 07:10:00+00:00,2026-01-01 07:40:00+00:00,-70.66,-33.45,-70.65,-33.44,88b2c55417fffff,88b2c5541bfffff
1,m1,u1,metro,study,12.5,True,2026-01-01 08:00:00+00:00,2026-01-01 08:35:00+00:00,-70.64,-33.46,-70.63,-33.45,88b2c5548dfffff,88b2c554c1fffff
2,m2,u2,car,work,1.2,False,2026-01-01 09:30:00+00:00,2026-01-01 10:00:00+00:00,-70.80,-33.60,-70.79,-33.59,88b2c54e11fffff,88b2c54525fffff
3,m3,u3,walk,None,0.4,False,2026-01-01 10:00:00+00:00,2026-01-01 10:20:00+00:00,-70.66,-33.45,-70.81,-33.61,88b2c55417fffff,88b2c54e3bfffff
4,m4,u4,bus,leisure,25.0,True,2026-01-01 11:00:00+00:00,2026-01-01 11:30:00+00:00,-70.90,-33.70,-70.91,-33.71,88b2c54da1fffff,88b2c0b29bfffff


,movement_id,user_id,mode,purpose,distance_km,is_peak,origin_time_utc,destination_time_utc,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index
0,m0,u0,bus,work,5.0,True,2026-01-01 07:10:00+00:00,2026-01-01 07:40:00+00:00,-70.66,-33.45,-70.65,-33.44,88b2c55417fffff,88b2c5541bfffff
1,m1,u1,metro,study,12.5,True,2026-01-01 08:00:00+00:00,2026-01-01 08:35:00+00:00,-70.64,-33.46,-70.63,-33.45,88b2c5548dfffff,88b2c554c1fffff
2,m2,u2,car,work,1.2,False,2026-01-01 09:30:00+00:00,2026-01-01 10:00:00+00:00,-70.80,-33.60,-70.79,-33.59,88b2c54e11fffff,88b2c54525fffff
3,m3,u3,walk,None,0.4,False,2026-01-01 10:00:00+00:00,2026-01-01 10:20:00+00:00,-70.66,-33.45,-70.81,-33.61,88b2c55417fffff,88b2c54e3bfffff
4,m4,u4,bus,leisure,25.0,True,2026-01-01 11:00:00+00:00,2026-01-01 11:30:00+00:00,-70.90,-33.70,-70.91,-33.71,88b2c54da1fffff,88b2c0b29bfffff


[Issue(level='error', code='FLT.WHERE.FIELD_NOT_FOUND', message="No se puede aplicar la cláusula where sobre 'does_not_exist': la columna no existe en trips.data.", field='does_not_exist', source_field=None, row_count=None, details={'check': 'where', 'field': 'does_not_exist', 'op': None, 'clause_shape': 'scalar', 'op_value': 'x', 'op_value_type': 'str', 'available_fields_sample': ['movement_id', 'user_id', 'mode', 'purpose', 'distance_km', 'is_peak', 'origin_time_utc', 'destination_time_utc', 'origin_longitude', 'origin_latitude'], 'available_fields_total': 14, 'allowed_ops': ['between', 'eq', 'gt', 'gte', 'in', 'is_null', 'lt', 'lte', 'ne', 'not_in', 'not_null'], 'dtype_effective': None, 'expected': 'existing column in trips.data', 'observed': 'does_not_exist', 'reason': 'field_not_found', 'action': 'omit_field_clause'}),
 Issue(level='warning', code='FLT.LIMIT.ISSUES_TRUNCATED', message='El reporte alcanzó max_issues=2; los hallazgos adicionales fueron truncados.', field=None, sourc